In [2]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_auc_score
import pandas as pd
import numpy as np
import scipy.sparse as sp
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    OneHotEncoder,
    MinMaxScaler,
    FunctionTransformer,
)
import category_encoders as ce
import random
from collections import Counter

In [3]:
df = pd.read_csv('data/training.csv')
df

,RefId,IsBadBuy,PurchDate,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,...,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,PRIMEUNIT,AUCGUART,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost
0,1,0,12/7/2009,ADESA,2006,3,MAZDA,MAZDA3,i,4D SEDAN I,...,11597.0,12409.0,NaN,NaN,21973,33619,FL,7100.0,0,1113
1,2,0,12/7/2009,ADESA,2004,5,DODGE,1500 RAM PICKUP 2WD,ST,QUAD CAB 4.7L SLT,...,11374.0,12791.0,NaN,NaN,19638,33619,FL,7600.0,0,1053
2,3,0,12/7/2009,ADESA,2005,4,DODGE,STRATUS V6,SXT,4D SEDAN SXT FFV,...,7146.0,8702.0,NaN,NaN,19638,33619,FL,4900.0,0,1389
3,4,0,12/7/2009,ADESA,2004,5,DODGE,NEON,SXT,4D SEDAN,...,4375.0,5518.0,NaN,NaN,19638,33619,FL,4100.0,0,630
4,5,0,12/7/2009,ADESA,2005,4,FORD,FOCUS,ZX3,2D COUPE ZX3,...,6739.0,7911.0,NaN,NaN,19638,33619,FL,4000.0,0,1020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72978,73010,1,12/2/2009,ADESA,2001,8,MERCURY,SABLE,GS,4D SEDAN GS,...,4836.0,5937.0,NaN,NaN,18111,30212,GA,4200.0,0,993
72979,73011,0,12/2/2009,ADESA,2007,2,CHEVROLET,MALIBU 4C,LS,4D SEDAN LS,...,10151.0,11652.0,NaN,NaN,18881,30212,GA,6200.0,0,1038
72980,73012,0,12/2/2009,ADESA,2005,4,JEEP,GRAND CHEROKEE 2WD V,Lar,4D WAGON LAREDO,...,11831.0,14402.0,NaN,NaN,18111,30212,GA,8200.0,0,1893
72981,73013,0,12/2/2009,ADESA,2006,3,CHEVROLET,IMPALA,LS,4D SEDAN LS,...,10099.0,11228.0,NaN,NaN,18881,30212,GA,7000.0,0,1974


In [4]:
target = 'IsBadBuy'
date_col = 'PurchDate'
df[date_col] = pd.to_datetime(df[date_col])
unique_dates = np.sort(df[date_col].unique())

n_dates = len(unique_dates)
train_end_date = unique_dates[int(n_dates * 0.33)]
valid_end_date = unique_dates[int(n_dates * 0.66)]

unique_dates = np.sort(df[date_col].unique())

n_dates = len(unique_dates)
train_end_date = unique_dates[int(n_dates * 0.33)]
valid_end_date = unique_dates[int(n_dates * 0.66)]

train_mask = df[date_col] < train_end_date
valid_mask = (df[date_col] >= train_end_date) & (df[date_col] < valid_end_date)
test_mask  = df[date_col] >= valid_end_date

train_df = df.loc[train_mask].copy()
valid_df = df.loc[valid_mask].copy()
test_df  = df.loc[test_mask].copy()

X_train = train_df.drop(columns=[target])
y_train = train_df[target]

X_valid = valid_df.drop(columns=[target])
y_valid = valid_df[target]

X_test = test_df.drop(columns=[target])
y_test = test_df[target]

drop_cols = ['RefId', 'PurchDate']
X_train = X_train.drop(columns=drop_cols)
X_valid = X_valid.drop(columns=drop_cols)
X_test  = X_test.drop(columns=drop_cols)


cat_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
num_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', MinMaxScaler())
])


cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', ce.CountEncoder(
        normalize=True, handle_unknown=0, handle_missing=0
    ))
])


preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, num_features),
        ('cat', cat_pipeline, cat_features)
    ],
    remainder='drop'
)

X_train = preprocessor.fit_transform(X_train)
X_valid = preprocessor.transform(X_valid)
X_test  = preprocessor.transform(X_test)
X_train = np.array(X_train)
y_train = np.array(y_train)
X_valid = np.array(X_valid)
y_valid = np.array(y_valid)
X_test = np.array(X_test)
y_test = np.array(y_test)
print(len(X_train), len(X_valid), len(X_test))

22772 23911 26300


In [ ]:
class MLP:
    def __init__(self, n_hidden=100, activation='tanh', lr=0.01,
                 epochs=100, batch_size=32, optimizer='sgd'):
        self.n_hidden = n_hidden
        self.lr = lr
        self.epochs = epochs
        self.batch_size = batch_size
        self.optimizer = optimizer

        if callable(activation):
            self.activation = activation
            self.activation_deriv = lambda x: (activation(x + 1e-7) - activation(x - 1e-7)) / (2e-7)
        elif activation == 'tanh':
            self.activation = np.tanh
            self.activation_deriv = lambda x: 1 - np.tanh(x) ** 2
        elif activation == 'arctan':
            self.activation = np.arctan
            self.activation_deriv = lambda x: 1 / (1 + x ** 2)
        elif activation == 'relu':
            self.activation = lambda x: np.maximum(0, x)
            self.activation_deriv = lambda x: (x > 0).astype(float)
        elif activation == 'sigmoid':
            self.activation = lambda x: 1 / (1 + np.exp(-np.clip(x, -500, 500)))
            self.activation_deriv = lambda x: self.activation(x) * (1 - self.activation(x))
        elif activation == 'cosine':
            self.activation = np.cos
            self.activation_deriv = lambda x: -np.sin(x)
        else:
            raise ValueError(f"Unknown activation: {activation}")

    def _sigmoid(self, x):
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

    def _initialize_parameters(self, n_features):
        rng = np.random.default_rng(42)
        self.W1 = rng.normal(0, 0.01, (n_features, self.n_hidden))
        self.b1 = np.zeros((1, self.n_hidden))
        self.W2 = rng.normal(0, 0.01, (self.n_hidden, 1))
        self.b2 = np.zeros((1, 1))

        if self.optimizer == 'adam':
            self._t = 0
            self._m = {k: np.zeros_like(getattr(self, k)) for k in ['W1', 'b1', 'W2', 'b2']}
            self._v = {k: np.zeros_like(getattr(self, k)) for k in ['W1', 'b1', 'W2', 'b2']}

    def _forward_pass(self, X):
        Z1 = X @ self.W1 + self.b1
        A1 = self.activation(Z1)
        Z2 = A1 @ self.W2 + self.b2
        A2 = self._sigmoid(Z2)
        return {'X': X, 'Z1': Z1, 'A1': A1, 'Z2': Z2, 'A2': A2}

    def _compute_loss(self, A2, y):
        eps = 1e-12
        y = y.reshape(-1, 1)
        return -np.mean(y * np.log(A2 + eps) + (1 - y) * np.log(1 - A2 + eps))

    def _backward_pass(self, cache, y):
        X, A1, Z1, A2 = cache['X'], cache['A1'], cache['Z1'], cache['A2']
        y = y.reshape(-1, 1)
        n_batch = len(y)

        dZ2 = (A2 - y) / n_batch
        dW2 = A1.T @ dZ2
        db2 = np.sum(dZ2, axis=0, keepdims=True)
        dA1 = dZ2 @ self.W2.T
        dZ1 = dA1 * self.activation_deriv(Z1)
        dW1 = X.T @ dZ1
        db1 = np.sum(dZ1, axis=0, keepdims=True)

        return {'W1': dW1, 'b1': db1, 'W2': dW2, 'b2': db2}

    def _update_sgd(self, grads):
        for k in ['W1', 'b1', 'W2', 'b2']:
            setattr(self, k, getattr(self, k) - self.lr * grads[k])

    def _update_adam(self, grads, beta1=0.9, beta2=0.999, eps=1e-8):
        self._t += 1
        t = self._t
        for k in ['W1', 'b1', 'W2', 'b2']:
            param = getattr(self, k)
            self._m[k] = beta1 * self._m[k] + (1 - beta1) * grads[k]
            self._v[k] = beta2 * self._v[k] + (1 - beta2) * (grads[k] ** 2)
            m_hat = self._m[k] / (1 - beta1 ** t)
            v_hat = self._v[k] / (1 - beta2 ** t)
            setattr(self, k, param - self.lr * m_hat / (np.sqrt(v_hat) + eps))

    def fit(self, X, y, X_val=None, y_val=None, verbose=True):
        n_samples, n_features = X.shape
        self._initialize_parameters(n_features)
        y = y.astype(float)
        if y_val is not None:
            y_val = y_val.astype(float)

        for epoch in range(self.epochs):
            idx = np.random.permutation(n_samples)
            X_s, y_s = X[idx], y[idx]
            epoch_losses = []

            for start in range(0, n_samples, self.batch_size):
                end = min(start + self.batch_size, n_samples)
                cache = self._forward_pass(X_s[start:end])
                epoch_losses.append(self._compute_loss(cache['A2'], y_s[start:end]))
                grads = self._backward_pass(cache, y_s[start:end])
                self._update_adam(grads) if self.optimizer == 'adam' else self._update_sgd(grads)

            if verbose and (epoch + 1) % 10 == 0:
                avg_loss = np.mean(epoch_losses)
                if X_val is not None and y_val is not None:
                    val_acc = np.mean(self.predict(X_val) == y_val)
                    print(f"Epoch {epoch+1}/{self.epochs} | Loss: {avg_loss:.4f} | Val Acc: {val_acc:.4f}")
                else:
                    print(f"Epoch {epoch+1}/{self.epochs} | Loss: {avg_loss:.4f}")

    def predict_proba(self, X):
        p1 = self._forward_pass(X)['A2'].flatten()
        return np.column_stack([1 - p1, p1])

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X)[:, 1] >= threshold).astype(int)


In [6]:
model = MLP(
    n_hidden=64,
    activation='relu',
    lr=0.001,
    epochs=100,
    batch_size=32,
    optimizer='adam'
)

model.fit(X_train, y_train, X_valid, y_valid)

Epoch 10/100 | Loss: 0.3313 | Val Acc: 0.8676
Epoch 20/100 | Loss: 0.3302 | Val Acc: 0.8676
Epoch 30/100 | Loss: 0.3300 | Val Acc: 0.8676
Epoch 40/100 | Loss: 0.3300 | Val Acc: 0.8676
Epoch 50/100 | Loss: 0.3298 | Val Acc: 0.8676
Epoch 60/100 | Loss: 0.3297 | Val Acc: 0.8676
Epoch 70/100 | Loss: 0.3293 | Val Acc: 0.8676
Epoch 80/100 | Loss: 0.3294 | Val Acc: 0.8676
Epoch 90/100 | Loss: 0.3296 | Val Acc: 0.8676
Epoch 100/100 | Loss: 0.3293 | Val Acc: 0.8676


In [7]:
valid_proba = model.predict_proba(X_valid)[:, 1]

valid_auc = roc_auc_score(y_valid, valid_proba)

valid_gini = 2 * valid_auc - 1

print("Validation AUC:", valid_auc)
print("Validation Gini:", valid_gini)

Validation AUC: 0.6638781811267525
Validation Gini: 0.3277563622535049


In [8]:
from sklearn.neural_network import MLPClassifier
sklearn_mlp = MLPClassifier(
    hidden_layer_sizes=(64,),   
    activation='relu',                
    solver='adam',                   
    alpha=0.0001,                     
    batch_size=32,                    
    learning_rate_init=0.001,         
    max_iter=100,                     
    random_state=42,                  
    verbose=True,                     
    early_stopping=False              
)
sklearn_mlp.fit(X_train, y_train)

valid_proba_sk = sklearn_mlp.predict_proba(X_valid)[:, 1]
valid_gini_sk = 2 * roc_auc_score(y_valid, valid_proba_sk) - 1
print(f"   Valid Gini: {valid_gini_sk:.4f}")

Iteration 1, loss = 0.34234427
Iteration 2, loss = 0.33245661
Iteration 3, loss = 0.33214069
Iteration 4, loss = 0.33104869
Iteration 5, loss = 0.33079988
Iteration 6, loss = 0.33029211
Iteration 7, loss = 0.33006689
Iteration 8, loss = 0.32979760
Iteration 9, loss = 0.32947637
Iteration 10, loss = 0.32946121
Iteration 11, loss = 0.32880871
Iteration 12, loss = 0.32889968
Iteration 13, loss = 0.32853543
Iteration 14, loss = 0.32817218
Iteration 15, loss = 0.32823967
Iteration 16, loss = 0.32742265
Iteration 17, loss = 0.32731586
Iteration 18, loss = 0.32719483
Iteration 19, loss = 0.32672790
Iteration 20, loss = 0.32640976
Iteration 21, loss = 0.32644008
Iteration 22, loss = 0.32622479
Iteration 23, loss = 0.32641392
Iteration 24, loss = 0.32573165
Iteration 25, loss = 0.32541556
Iteration 26, loss = 0.32552122
Iteration 27, loss = 0.32542290
Iteration 28, loss = 0.32472084
Iteration 29, loss = 0.32538430
Iteration 30, loss = 0.32474867
Iteration 31, loss = 0.32477616
Iteration 32, los

/usr/local/lib/python3.10/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(


   Valid Gini: 0.3542


## Пункт 4 — Сравнение: sklearn MLPClassifier vs мой MLP

В данном случае sklearn MLPClassifier показал **более высокий Gini**, чем мой MLP. Причины:

1. **Инициализация весов (Glorot/Xavier).** nn.Linear и sklearn оба используют Glorot-инициализацию, которая масштабирует веса с учётом числа входов и выходов слоя. Наш MLP инициализирует веса с фиксированным std=0.01 — это слишком маленькие веса для данного числа признаков, из-за чего градиенты в первых эпохах очень малы и обучение идёт медленнее.

2. **Реализация Adam.** sklearn использует оптимизированную реализацию adam из scipy. Наша реализация математически эквивалентна, но написана на чистом NumPy и может незначительно отличаться в численных деталях.

3. **Численная стабильность log-loss.** sklearn вычисляет log-loss с дополнительными защитами от переполнения, что стабилизирует обучение на несбалансированных данных (а IsBadBuy — несбалансированный таргет: ~12% положительных).

4. **Отсутствие early stopping.** Оба обучаются 100 эпох без ранней остановки — это одинаковое условие, но sklearn может эффективнее использовать каждую эпоху благодаря лучшей инициализации.


In [9]:
results = {}

for act in ['relu', 'sigmoid', 'cosine']:
    m = MLP(n_hidden=64, activation=act, lr=0.001, epochs=100,
            batch_size=32, optimizer='adam')
    m.fit(X_train, y_train, verbose=False)
    gini = 2 * roc_auc_score(y_valid, m.predict_proba(X_valid)[:, 1]) - 1
    results[act] = gini
    print(f"Activation: {act:8s} | Valid Gini: {gini:.4f}")

best_act = max(results, key=results.get)
print(f"\nBest activation: {best_act} with Gini = {results[best_act]:.4f}")


Activation: relu     | Valid Gini: 0.3275
Activation: sigmoid  | Valid Gini: 0.3224
Activation: cosine   | Valid Gini: 0.3508

Best activation: cosine with Gini = 0.3508


## Пункт 6 — PyTorch MLP

In [10]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

X_tr = torch.tensor(X_train, dtype=torch.float32)
X_vl = torch.tensor(X_valid, dtype=torch.float32)
X_ts = torch.tensor(X_test,  dtype=torch.float32)
y_tr = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)

train_dataset = TensorDataset(X_tr, y_tr)
train_loader  = DataLoader(train_dataset, batch_size=32, shuffle=True)

n_features = X_train.shape[1]
torch_model = nn.Sequential(
    nn.Linear(n_features, 64),
    nn.ReLU(),
    nn.Linear(64, 1),
    nn.Sigmoid()
)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(torch_model.parameters(), lr=1e-3)

N_EPOCHS = 100
for epoch in range(1, N_EPOCHS + 1):
    torch_model.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        pred = torch_model(xb)
        loss = criterion(pred, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(xb)

    if epoch % 10 == 0:
        torch_model.eval()
        with torch.no_grad():
            val_prob = torch_model(X_vl).numpy()
        val_gini = 2 * roc_auc_score(y_valid, val_prob) - 1
        avg_loss = total_loss / len(X_train)
        print(f"Epoch {epoch:3d}/{N_EPOCHS} | Loss: {avg_loss:.4f} | Val Gini: {val_gini:.4f}")

def gini(y_true, y_prob):
    return 2 * roc_auc_score(y_true, y_prob) - 1

torch_model.eval()
with torch.no_grad():
    train_prob_torch = torch_model(X_tr).numpy()
    valid_prob_torch = torch_model(X_vl).numpy()
    test_prob_torch  = torch_model(X_ts).numpy()

train_gini = gini(y_train, train_prob_torch)
valid_gini = gini(y_valid, valid_prob_torch)
test_gini  = gini(y_test,  test_prob_torch)

print(f"  Train Gini : {train_gini:.4f}")
print(f"  Valid Gini : {valid_gini:.4f}")
print(f"  Test  Gini : {test_gini:.4f}")
print(f"\n Valid - Test: {valid_gini - test_gini:+.4f}")


Epoch  10/100 | Loss: 0.3299 | Val Gini: 0.3268
Epoch  20/100 | Loss: 0.3277 | Val Gini: 0.3403
Epoch  30/100 | Loss: 0.3260 | Val Gini: 0.3543
Epoch  40/100 | Loss: 0.3252 | Val Gini: 0.3599
Epoch  50/100 | Loss: 0.3246 | Val Gini: 0.3592
Epoch  60/100 | Loss: 0.3238 | Val Gini: 0.3648
Epoch  70/100 | Loss: 0.3225 | Val Gini: 0.3684
Epoch  80/100 | Loss: 0.3213 | Val Gini: 0.3545
Epoch  90/100 | Loss: 0.3208 | Val Gini: 0.3550
Epoch 100/100 | Loss: 0.3202 | Val Gini: 0.3644
  Train Gini : 0.4413
  Valid Gini : 0.3644
  Test  Gini : 0.3645

 Valid - Test: -0.0001


## Пункт 6 — Сравнение PyTorch vs мой MLP

Преимущества PyTorch:
- **Автодифференцирование (autograd)** — градиенты вычисляются автоматически, нет риска ошибки в backprop вручную.
- **Оптимизированные операции** — PyTorch использует BLAS/LAPACK под капотом, что быстрее чистого NumPy на больших сетях.
- **GPU поддержка** — при наличии GPU можно ускорить обучение одной строкой
- **Инициализация Kaiming (He)** — nn.Linear по умолчанию использует инициализацию Kaiming, которая лучше подходит для ReLU.

## Пункт 7 — Финальная оценка лучшей модели (train / valid / test Gini)

Лучшая модель PyTorch MLP
- нет переобучения
